# Tool 返回巨大输出，模型必须全部读取吗？

## V0.5 Virtual Resource / Artifact Handle

这个 lab 展示大 payload、模型可见 handle、可按范围读取的 resource 三者不是一回事。

**Core:** Resource != Context. Handle != Resource. Handle != Permission.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Create a deterministic large artifact

In [ ]:
import tempfile
from pathlib import Path

from agentkernel import LocalResourceStore, ResourceOwner, ResourceService

payload = ("diagnostic line\n" * 2_000).encode()
owner = ResourceOwner("lab-v0-5-agent", "lab-v0-5-session")
tmpdir = tempfile.TemporaryDirectory(prefix="agentkernel-lab-v0-5-")
store_root = Path(tmpdir.name) / "resources"
service = ResourceService(LocalResourceStore(store_root))
handle = service.create_artifact(
    payload,
    owner=owner,
    media_type="text/plain",
    encoding="utf-8",
    source_tool_name="logs.collect",
    source_tool_call_id="call-logs-1",
    source_operation_id="op-logs-1",
)
print_table([
    {"fact": "handle uri", "value": handle.uri},
    {"fact": "model-visible marker bytes", "value": len(handle.uri)},
    {"fact": "resource bytes", "value": handle.size_bytes},
])

## 2. Read selected ranges instead of placing all bytes in context

In [ ]:
first = service.read(handle.uri, owner=owner, offset=0, limit=32)
second = service.read(handle.uri, owner=owner, offset=64, limit=32)
print_table([
    {"range": "0..32", "preview": first.data.decode(), "has_more": first.has_more},
    {"range": "64..96", "preview": second.data.decode(), "has_more": second.has_more},
])

## 3. Restart ResourceService and read again from the same store

In [ ]:
restarted = ResourceService(LocalResourceStore(store_root))
again = restarted.read(handle.uri, owner=owner, offset=0, limit=32)
print_table([
    {"fact": "restart read success", "value": again.data == payload[:32]},
    {"fact": "resource metrics are service-local", "value": restarted.metrics.resource_reads},
])
trajectory("Tool returns large bytes", "ResourceService stores bytes", "Context receives artifact:// handle", "Reader asks for byte range")
tmpdir.cleanup()

## Invariant

The model can carry a bounded reference while the full bytes live outside context.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- Resource handles keep large bytes out of model context.
- Range reads make inspection explicit and bounded.
- Restart can read stored artifacts from the same local store.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not make a handle into permission.
- It does not prove production storage durability.
- It does not use a real model provider.